# Module 7 • In‑Class Lab: Sentiment Analysis


This notebook demonstrates Sentiment Analysis using `VADER`, `Linear SVM`, and `transformers pipeline`. It uses a tiny dataset:
- `tiny_tweets.csv`


You'll complete short tasks to:
1. Load data and explore.
2. Run **VADER** for quick rule-based sentiment.
3. Train a **Linear SVM** with TF‑IDF (classical ML).
4. Compare with a **transformers** pipeline.

**Deliverables (end of class):**
- Confusion matrix + precision/recall/F1 for SVM on tweets.
- One short paragraph: Which approach performed better and why?


## 0. Setup


In [0]:
## Uncomment if needed
#!pip install nltk scikit-learn matplotlib transformers --upgrade

import pandas as pd
import numpy as np
from pathlib import Path
import nltk
SEED = 42
rng = np.random.default_rng(SEED)


## Task 1. Load data & quick EDA

In [0]:
tweets = pd.read_csv('tweets_300.csv')
display(tweets.head())
tweets['label'].value_counts()

text,label
dear @verizonsupport your service is so poor in dallas... no internet at all.,negative
@verizonsupport ive sent you a dm,neutral
thanks to michelle at @verizonsupport who solved my issue. great help!,positive
this update ruined my battery life,negative
shipping was fast and the phone works perfectly,positive


neutral     103
negative    102
positive     95
Name: label, dtype: int64

### Task 1.1
- Compute the average tweet length (in characters & tokens).
- Print a class distribution table for `tweets`.

In [0]:
## Your code here
avg_chars = tweets['text'].str.len().mean()
avg_tokens = tweets['text'].str.split().apply(len).mean()
print({'avg_chars': avg_chars, 'avg_tokens': avg_tokens})
tweets['label'].value_counts(normalize=True)

{'avg_chars': 27.19333333333333, 'avg_tokens': 4.46}


neutral     0.343333
negative    0.340000
positive    0.316667
Name: label, dtype: float64

##Task 2. Develop Rule‑based baseline with VADER
VADER stands for Valence Aware Dictionary and sEntiment Reasoner. It’s a rule-based sentiment analysis tool specifically designed to analyze social media text, but it works well on other types of text too.

Optional: Just an exercise, how to get the score of a specific word in VADER

In [0]:
#1: Import and Download VADER Lexicon
nltk.download('vader_lexicon')

#Initialize VADER Sentiment Analyzer
from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

# Make sure the lexicon is downloaded
nltk.download('vader_lexicon')

# Load the analyzer
sia = SentimentIntensityAnalyzer()

# Access the lexicon dictionary
vader_lexicon = sia.lexicon

# Example: Get the score for the word "waste"
print(vader_lexicon.get("great"))
print(vader_lexicon.get("waste"))


3.1
-1.8


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/spark-2de3a460-893f-4e53-9396-bb/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/spark-2de3a460-893f-4e53-9396-bb/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


### How VADER works?

### VADER Sentiment Analysis – Compound Score Example

#### Sentence:
**"I absolutely love it!"**



#### Step-by-Step Calculation

1. **Valence Scores from VADER Lexicon**:
   - `"love"` → **3.2**
   - `"absolutely"` → intensifier (boosts nearby sentiment)
   - `"it"` → neutral (ignored)

2. **Adjusted Score**:
   - `"absolutely"` intensifies `"love"` slightly.
   - Final adjusted score for `"love"` ≈ **3.4**

3. **Valence Sum**:
   - Total = **3.4**

4. **Valence Squared Sum**:
   - $$3.4^2 = 11.56$$

5. **Normalization Formula**:
   VADER uses:
   $$
   \text{compound} = \frac{\sum \text{valence}}{\sqrt{\sum \text{valence}^2 + 15}} = \frac{3.4}{\sqrt{11.56 + 15}} = \frac{3.4}{\sqrt{26.56}} ≈ \frac{3.4}{5.15} ≈ 0.66
   $$
This is a postive sentiment sinc erthe score is > 0.05



**Back to our exercise**

In [0]:
#1: Import and Download VADER Lexicon
import nltk
nltk.download('vader_lexicon')

#2: Initialize VADER Sentiment Analyzer
from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

#3: Define Sentiment Labeling Function
def vader_label(text, pos=0.05, neg=-0.05):
    """
    Classifies sentiment based on VADER compound score.
    - Compound score ranges from -1 (most negative) to +1 (most positive).
    - Thresholds:
        >= 0.05 → positive
        <= -0.05 → negative
        else → neutral
    """
    c = sia.polarity_scores(text)['compound']
    return 'positive' if c >= pos else ('negative' if c <= neg else 'neutral')

#4: Apply VADER to Tweet Texts
# Assumes 'tweets' DataFrame contains a 'text' column and a 'label' column
tweets['vader_pred'] = tweets['text'].apply(vader_label)

#5: Display Results
# Shows original tweet, true label, and VADER prediction
tweets[['text', 'label', 'vader_pred']]


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/spark-2de3a460-893f-4e53-9396-bb/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


,text,label,vader_pred
0,dear @verizonsupport your service is so poor i...,negative,negative
1,@verizonsupport ive sent you a dm,neutral,neutral
2,thanks to michelle at @verizonsupport who solv...,positive,positive
3,this update ruined my battery life,negative,negative
4,shipping was fast and the phone works perfectly,positive,positive
...,...,...,...
295,fast delivery and great quality,positive,positive
296,trying the app now,neutral,neutral
297,not worth the money,negative,negative
298,app is full of bugs,negative,neutral


### Task 2.1 VADER Evaluation
- Compute accuracy, precision, recall, and F1 for VADER.
- Print the confusion matrix.

In [0]:
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(tweets['label'], tweets['vader_pred']))
print(confusion_matrix(tweets['label'], tweets['vader_pred']))

              precision    recall  f1-score   support

    negative       0.84      0.60      0.70       102
     neutral       0.68      0.63      0.66       103
    positive       0.72      1.00      0.84        95

    accuracy                           0.74       300
   macro avg       0.75      0.74      0.73       300
weighted avg       0.75      0.74      0.73       300

[[61 30 11]
 [12 65 26]
 [ 0  0 95]]


## Task 3. Develop Classical ML: Linear SVM with TF‑IDF
We'll do a simple train/test split on tweets and train a LinearSVC and evaluate the developed model.

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

X_train, X_test, y_train, y_test = train_test_split(
    tweets['text'], tweets['label'], test_size=0.3, random_state=SEED, stratify=tweets['label']
)
tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=1)
Xtr = tfidf.fit_transform(X_train)
Xte = tfidf.transform(X_test)
clf = LinearSVC()
clf.fit(Xtr, y_train)
svm_pred = clf.predict(Xte)
print(classification_report(y_test, svm_pred))
print(confusion_matrix(y_test, svm_pred))

              precision    recall  f1-score   support

    negative       0.94      0.97      0.95        31
     neutral       0.97      0.97      0.97        31
    positive       0.96      0.93      0.95        28

    accuracy                           0.96        90
   macro avg       0.96      0.95      0.96        90
weighted avg       0.96      0.96      0.96        90

[[30  1  0]
 [ 0 30  1]
 [ 2  0 26]]


## 4. Transformer pipeline
Compare predictions using a pretrained model (e.g., `distilbert-base-uncased-finetuned-sst-2-english`).


## Hugging Face and Sentiment Analysis

**Hugging Face**, https://huggingface.co/,  is a leading company in Natural Language Processing (NLP), providing an open-source **Transformers library** with access to numerous pretrained models and datasets. These models enable state-of-the-art performance on tasks like sentiment analysis, summarization, translation, and question answering, without requiring deep learning expertise.

### Key Models
- **BERT**: A deep bidirectional transformer that understands context by reading text both left-to-right and right-to-left.
- **RoBERTa**: An optimized version of BERT, trained longer on more data without Next Sentence Prediction, achieving higher accuracy.
- **DistilBERT**: A smaller, faster version of BERT using model compression, retaining most of its performance.

### Sentiment Analysis with Hugging Face
- Hugging Face offers numerous models fine-tuned for **sentiment classification** across domains like movie reviews, product feedback, and social media.
- Models support:
  - **Binary classification** (e.g., positive vs. negative)
  - **Multi-class sentiment** (e.g., star ratings, neutral tone)
- These models leverage deep contextual understanding, making them effective for:
  - Informal or short text
  - Multilingual scenarios
- Applications include:
  - Business intelligence
  - Social media monitoring
  - User feedback analysis

Hugging Face pipelines allow real-time processing of large volumes of text for actionable insights.



## Common Hugging Face Sentiment Analysis Models

| **Model Name**                                      | **Type**            | **Sentiment Classes**           | **Trained On**                  | **Best For**                          |
|------------------------------------------------------|----------------------|----------------------------------|----------------------------------|----------------------------------------|
| `distilbert-base-uncased-finetuned-sst-2-english`   | DistilBERT          | Positive / Negative             | SST-2 (Stanford)                | General English sentiment             |
| `cardiffnlp/twitter-roberta-base-sentiment`         | RoBERTa             | Positive / Neutral / Negative   | 60M English Tweets              | Social media & Twitter                |
| `nlptown/bert-base-multilingual-uncased-sentiment`  | Multilingual BERT   | 1–5 star ratings                | Amazon, Yelp, Trustpilot        | Reviews & multilingual texts          |
| `siebert/sentiment-roberta-large-english`           | RoBERTa Large       | Positive / Negative             | IMDb, Yelp, Amazon              | High-accuracy English sentiment       |
| `finiteautomata/bertweet-base-sentiment-analysis`   | BERTweet            | Positive / Neutral / Negative   | Twitter sentiment corpus        | Tweets & social media slang           |


In [0]:
pip install torch

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
#dbutils.library.restartPython()

In [0]:

USE_TRANSFORMER = True  # Set False if you cannot download models

if USE_TRANSFORMER:
    from transformers import pipeline
    from sklearn.metrics import classification_report

    # Choose your model (uncomment one if needed)
    model_name = 'distilbert-base-uncased-finetuned-sst-2-english'
    # model_name = 'finiteautomata/bertweet-base-sentiment-analysis'
    # model_name = 'cardiffnlp/twitter-roberta-base-sentiment'

    # Load Hugging Face pipeline
    clf_hf = pipeline(task='sentiment-analysis', model=model_name)

    # Predict sentiment for all tweets
    preds = [clf_hf(text)[0]['label'].lower() for text in tweets['text']]

    # Map predictions to {positive, negative, neutral}
    mapped = ['positive' if 'pos' in p else 'negative' if 'neg' in p else 'neutral' for p in preds]

    # Print performance report
    print(f"Model: {model_name}")
    print(classification_report(tweets['label'], mapped))
else:
    print('Transformer section skipped (set USE_TRANSFORMER=True to run).')


Device set to use cpu


Model: distilbert-base-uncased-finetuned-sst-2-english
              precision    recall  f1-score   support

    negative       0.71      1.00      0.83       102
     neutral       0.00      0.00      0.00       103
    positive       0.61      1.00      0.76        95

    accuracy                           0.66       300
   macro avg       0.44      0.67      0.53       300
weighted avg       0.43      0.66      0.52       300



/local_disk0/.ephemeral_nfs/envs/pythonEnv-2de3a460-893f-4e53-9396-bb29f2a9c737/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/local_disk0/.ephemeral_nfs/envs/pythonEnv-2de3a460-893f-4e53-9396-bb29f2a9c737/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/local_disk0/.ephemeral_nfs/envs/pythonEnv-2de3a460-893f-4e53-9396-bb29f2a9c737/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 

## Reflection: Short write‑up
Which approach (VADER vs. SVM vs. Transformer) worked best on this tiny set? Why might that change on a larger, noisier corpus?